We load the class `RecursiveSet`, the record classes representing literals, and the implementation of the Davis-Putnam algorithm that uses the Jeroslow-Wang heuristic.  The last file provides the functions `solve` and `first`.

In [ ]:
%load ../lib/RecursiveSet.jsh
%load Formula.jsh
%load 07-Davis-Putnam-JW.jsh

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

In [ ]:
RecursiveSet<Integer> range(int start, int stop) {
    RecursiveSet<Integer> result = new RecursiveSet<>();
    for (int i = start; i <= stop; ++i) {
        result.add(i);
    }
    return result;
}

## Sudoku

The Finnish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is shown below.
<img src="sudoku.png" alt="A game of Sudoku" width="50%">

The function `createPuzzle` returns the Sudoku as a two-dimensional array.  This array contains the number `0` for those cells that are left unspecified.

In [ ]:
int[][] createPuzzle() {
    return new int[][] {
        { 8, 0, 0, 0, 0, 0, 0, 0, 0 },
        { 0, 0, 3, 6, 0, 0, 0, 0, 0 },
        { 0, 7, 0, 0, 9, 0, 2, 0, 0 },
        { 0, 5, 0, 0, 0, 7, 0, 0, 0 },
        { 0, 0, 0, 0, 4, 5, 7, 0, 0 },
        { 0, 0, 0, 1, 0, 0, 0, 3, 0 },
        { 0, 0, 1, 0, 0, 0, 0, 6, 8 },
        { 0, 0, 8, 5, 0, 0, 0, 1, 0 },
        { 0, 9, 0, 0, 0, 0, 4, 0, 0 }
    };
}

We will solve this Sudoku using the Davis-Putnam algorithm.  We use the following variables:
* `Q<r,c,d>` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.
    
The function `varName(row, col, digit)` returns a formated string that is interpreted as a variable name.

In [ ]:
String varName(int row, int col, int digit) {
    return "Q<" + row + "," + col + "," + digit + ">";
}

In [ ]:
varName(1, 2, 3)

The function `atMostOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that at most one of the variables of `S` is true.
$$ \texttt{atMostOne}(S) = \bigl\{ \{\neg p, \neg q\} \bigm| \langle p, q\rangle \in S \times S \wedge p \not= q\bigr\}. $$

In [ ]:
RecursiveSet<RecursiveSet<Literal>> atMostOne(RecursiveSet<String> S) {
    return flatMap(S, p -> S.filterMap(q -> !p.equals(q),
                                       q -> set(new Neg(p), new Neg(q))));
}

The function `atLeastOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that at least one of the variables of `S` is true.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> atLeastOne(RecursiveSet<String> S) {
    return set(S.map(p -> new Var(p)));
}

The function `exactlyOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that exactly one of the variables of `S` is true.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> exactlyOne(RecursiveSet<String> S) {
    return atMostOne(S).union(atLeastOne(S));
}

In [ ]:
exactlyOne(set("a", "b", "c"))

A *position* is a pair of coordinates of the form `(row, col)` where
`row` specifies the row and `col` specifies the column.  Positions are represented as objects of type `Pair<Integer, Integer>`.  The record class `Pair` is defined in the file `RecursiveSet.jsh`.

The function `exactlyOneForDigit` takes two arguments:
* A set `L` of pairs of indices representing Sudoku coordinates as its first argument.
  The elements of `L` are pairs of the form `(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
* A specific `digit` as its second argument.
  `digit` is an element from the set `{1,...,9}`.

It returns a set of clauses expressing the fact that the given `digit`
appears exactly once among the cells specified by the coordinate pairs in `L`.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> exactlyOneForDigit(RecursiveSet<Pair<Integer, Integer>> L, int digit) {
    return exactlyOne(L.map(pos -> varName(pos.first(), pos.second(), digit)));
}

The function `exactlyOneForEveryDigit` takes a set `L` of pairs of indices as its argument.  The elements of `L` are pairs of the form
`(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
It returns a set of clauses expressing that each of the nine digits occurs at exactly one position in the set `L`.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> exactlyOneForEveryDigit(RecursiveSet<Pair<Integer, Integer>> L) {
    return flatMap(range(1, 9), d -> exactlyOneForDigit(L, d));
}

The function `exactlyOneDigit(row, col)` takes integers `row` and `col` as arguments.  These specify the row and column of a field in a Sudoku.  The function returns a set of clauses specifying that exactly one of the variables

* `Q<row,col,1>`, `Q<row,col,2>`, $\cdots$, `Q<row,col,9>`

is `true`.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> exactlyOneDigit(int row, int col) {
    return exactlyOne(range(1, 9).map(d -> varName(row, col, d)));
}

The function `constraintsFromPuzzle`  returns a set of clauses stating that the variables corresponding to numbers that are already given in the Sudoku puzzle take the values that are specified.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> constraintsFromPuzzle() {
    var Puzzle  = createPuzzle();
    var Indices = range(1, 9);
    return Indices.cartesianProduct(Indices).filterMap(
        pos -> Puzzle[pos.first() - 1][pos.second() - 1] != 0,
        pos -> set(new Var(varName(pos.first(), pos.second(),
                                   Puzzle[pos.first() - 1][pos.second() - 1]))));
}

In [ ]:
for (var clause : constraintsFromPuzzle()) {
    System.out.println(clause);
}
constraintsFromPuzzle().size()

The function `getBlockCells` calculates the absolute grid coordinates for all 9 cells 
within a specific 3x3 Sudoku block.  The Sudoku board is divided into a 3x3 grid of blocks. 
This function takes the zero-based indices of a block and maps its local cells to the 
1-based coordinates used by the solver variables. 
 * `r` - The row index of the 3x3 macro-block (must be 0, 1, or 2).
 * `c` - The column index of the 3x3 macro-block (must be 0, 1, or 2).
The function returns an array of 9 tuples, where each tuple `[row, col]` represents 
 the 1-based row and column coordinates on the full 9x9 Sudoku board.
 
For example, the set of coordinates for the top-middle block (`r = 0, c = 1`) 
is given as follows:
 ```getBlockCells(0, 1) = 
         [[1, 4], [1, 5], [1, 6], 
          [2, 4], [2, 5], [2, 6], 
          [3, 4], [3, 5], [3, 6]]
```

In [ ]:
RecursiveSet<Pair<Integer, Integer>> getBlockCells(int r, int c) {
    return range(1, 3).cartesianProduct(range(1, 3)).map(
        pos -> new Pair<>(r * 3 + pos.first(), c * 3 + pos.second())
    );
}

In [ ]:
getBlockCells(0, 1)

The function `allConstraints` returns a set of clauses that encodes the given sudoku.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> allConstraints() {
    // 1. Start with the constraints from the puzzle
    var Clauses = constraintsFromPuzzle();
    // 2. There is exactly one digit in every field
    Clauses.addAll(flatMap(
        range(1, 9).cartesianProduct(range(1, 9)),
        pos -> exactlyOneDigit(pos.first(), pos.second())
    ));
    // 3. All entries in a row are unique
    Clauses.addAll(flatMap(
        range(1, 9),
        row -> exactlyOneForEveryDigit(range(1, 9).map(col -> new Pair<>(row, col)))
    ));
    // 4. All entries in a column are unique
    Clauses.addAll(flatMap(
        range(1, 9),
        col -> exactlyOneForEveryDigit(range(1, 9).map(row -> new Pair<>(row, col)))
    ));
    // 5. All entries in a 3x3 square are unique
    var block = range(0, 2).cartesianProduct(range(0, 2));
    Clauses.addAll(flatMap(block, pos -> exactlyOneForEveryDigit(getBlockCells(pos.first(), pos.second()))));
    return Clauses;
}

In [ ]:
allConstraints().size()

The function `sudoku` solves the Sudoku.  If the Sudoku is not solvable, it returns `null`.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> sudoku() {
    var Clauses  = allConstraints();
    var Solution = solve(Clauses);
    RecursiveSet<Literal> EmptyClause = set();
    if (Solution.contains(EmptyClause)) {
        System.out.println("The problem is not solvable!");
        return null;
    }
    return Solution;
}

On my 2022 Mac Studio this takes about 15 minutes.

In [ ]:
long start = System.currentTimeMillis();
var Solution = sudoku();
System.out.println("sudoku: " + (System.currentTimeMillis() - start) / 1000.0 + " s");

In [ ]:
Solution

## Graphical Representation

The function `transformSolution` takes a solution and returns a two-dimensional array that contains the digits of the solved Sudoku.  The names of the variables have the form `Q<r,c,d>`.  The function strips the first two characters and the last character of a variable name and splits the remaining string at the commas.

In [ ]:
int[][] transformSolution(RecursiveSet<RecursiveSet<Literal>> Solution) {
    int[][] Result = new int[9][9];
    for (var UnitClause : Solution) {
        if (first(UnitClause) instanceof Var(var name)) {
            var parts = name.substring(2, name.length() - 1).split(",");
            int row   = Integer.parseInt(parts[0]);
            int col   = Integer.parseInt(parts[1]);
            int digit = Integer.parseInt(parts[2]);
            Result[row - 1][col - 1] = digit;
        }
    }
    return Result;
}

The function `showSolution` displays the solution as an HTML table.  The digits that are given in the puzzle are shown in bold.

In [ ]:
void showSolution(RecursiveSet<RecursiveSet<Literal>> Solution, String width) {
    var solution = transformSolution(Solution);
    var puzzle   = createPuzzle();
    var html     = new StringBuilder();
    html.append("""
        <table style="width:%s; border-collapse: collapse; border: 2px solid black; font-family: sans-serif;">
        """.formatted(width));
    for (int row = 0; row < 9; row++) {
        html.append("<tr>");
        for (int col = 0; col < 9; col++) {
            int    value     = solution[row][col];
            String cellStyle = "font-weight: normal;";
            if (puzzle[row][col] != 0) {
                value     = puzzle[row][col];
                cellStyle = "font-weight: bold;";
            }
            boolean isGray      = (row / 3 + col / 3) % 2 != 0;
            String  bgColor     = isGray ? "#f0f0f0" : "#ffffff";
            String  borderStyle = "border: 1px solid #ccc;";
            if ((col + 1) % 3 == 0 && col < 8) {
                borderStyle += "border-right: 2px solid black;";
            }
            if ((row + 1) % 3 == 0 && row < 8) {
                borderStyle += "border-bottom: 2px solid black;";
            }
            html.append("""
                <td style="%s width:30px; height:30px; text-align:center; font-size:20px; \
                background-color:%s; %s">%s</td>""".formatted(
                    borderStyle, bgColor, cellStyle, value != 0 ? value : ""));
        }
        html.append("</tr>");
    }
    html.append("</table>");
    display(html.toString(), "text/html");
}

void showSolution(RecursiveSet<RecursiveSet<Literal>> Solution) {
    showSolution(Solution, "50%");
}

In [ ]:
showSolution(Solution);